In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


from catboost import CatBoostClassifier

RANDOM_STATE = 42
TARGET_COL = "flag"

In [10]:
df = pd.read_pickle("../data/processed/dataset_with_target.pkl")
df.shape, TARGET_COL in df.columns

((3000000, 168), True)

In [18]:
df_small, _ = train_test_split(
    df,
    train_size=300_000,
    stratify=df[TARGET_COL],
    random_state=RANDOM_STATE
)

df_small.shape


(300000, 168)

In [19]:
df_raw = pd.read_parquet(
    "../data/raw/train_data_0.pq",
        columns=[
        "id",
        "enc_loans_credit_status",
        "enc_loans_account_holder_type",
    ])

df_raw.shape

(1974724, 3)

In [20]:
ids = df_small[["id"]]

df_raw_small = df_raw.merge(ids, on="id", how="inner")

df_raw_small.shape


(196739, 3)

In [22]:
ohe_credit_status = (
    pd.get_dummies(
        df_raw_small[["id", "enc_loans_credit_status"]],
        columns=["enc_loans_credit_status"],
        prefix="enc_loans_credit_status"
    )
    .groupby("id", as_index=False)
    .mean()
)

ohe_credit_status.shape



(24942, 8)

In [23]:
ohe_holder_type = (
    pd.get_dummies(
        df_raw_small[["id", "enc_loans_account_holder_type"]],
        columns=["enc_loans_account_holder_type"],
        prefix="enc_loans_account_holder_type"
    )
    .groupby("id", as_index=False)
    .mean()
)

ohe_holder_type.shape



(24942, 7)

In [24]:
df_small_full = (
    df_small
    .merge(ohe_credit_status, on="id", how="left")
    .merge(ohe_holder_type, on="id", how="left")
)

df_small_full.shape



(300000, 181)

In [26]:
[c for c in df_small_full.columns if c.startswith("enc_loans_credit_status_")],
[c for c in df_small_full.columns if c.startswith("enc_loans_account_holder_type_")]



['enc_loans_account_holder_type_1',
 'enc_loans_account_holder_type_2',
 'enc_loans_account_holder_type_3',
 'enc_loans_account_holder_type_4',
 'enc_loans_account_holder_type_5',
 'enc_loans_account_holder_type_6']

In [27]:
X = df_small_full.drop(columns=[TARGET_COL])
y = df_small_full[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)


In [28]:
model = CatBoostClassifier(
    iterations=600,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=100
)

model.fit(X_train, y_train)


0:	total: 265ms	remaining: 2m 38s
100:	total: 10.1s	remaining: 50.1s
200:	total: 17.3s	remaining: 34.4s
300:	total: 27.1s	remaining: 26.9s
400:	total: 40.8s	remaining: 20.3s
500:	total: 50.6s	remaining: 10s
599:	total: 57.8s	remaining: 0us


In [29]:
y_pred = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)

auc


0.7202854011341968